# Top interactors by interface metrics
Load every `results/<bait>__<prey>/interface_metrics.parquet`, rank preys, and merge with `data/pxd013906_uniprot_mapping.csv`.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("..") if Path("../results").exists() else Path(".")
RANK_METRIC = "ipsae"   # any of: ipsae, ilis, pinc, pdockq2, pair_iptm, ...
TOP_N = 15

In [ ]:
files = sorted((ROOT / "results").glob("*/interface_metrics.parquet"))
metrics = pd.concat((pd.read_parquet(f) for f in files), ignore_index=True)
metrics[["bait", "prey"]] = metrics["pair"].str.split("__", n=1, expand=True)
print(len(files), "pairs;", len(metrics), "rows")
metrics.head()

## Rank preys
Per pair: best value of the ranking metric over diffusion samples and chain pairs. Then take the top N.

In [ ]:
per_pair = (metrics.groupby(["pair", "bait", "prey"], as_index=False)
            .agg(best=(RANK_METRIC, "max"),
                 mean=(RANK_METRIC, "mean"),
                 n_rows=(RANK_METRIC, "size"),
                 ilis=("ilis", "max"), pinc=("pinc", "max"),
                 pdockq2=("pdockq2", "max"), pair_iptm=("pair_iptm", "max")))
top = per_pair.sort_values("best", ascending=False).head(TOP_N).reset_index(drop=True)
top

## Merge with UniProt info table

In [ ]:
info = pd.read_csv(ROOT / "data" / "pxd013906_uniprot_mapping.csv")
info = info.drop_duplicates("uniprot")   # one row per UniProt id
merged = top.merge(info, left_on="prey", right_on="uniprot", how="left")
print("unmatched preys:", merged["uniprot"].isna().sum())
merged